# DIEM Chatbot — Costruzione del Vector Database

> **Versione intermedia di selezione.**  
> Questo notebook contiene esclusivamente le celle del notebook originale che appartengono al flusso principale: crawling, estrazione strutturata, chunking, metadati specifici per sezione e popolamento di Qdrant.  
> Il contenuto delle celle non è stato ancora rifattorizzato: verrà ripulito e semplificato nei passaggi successivi.


# Installazioni necessarie

In [ ]:
# ============================================================
# INSTALLAZIONE DELLE DIPENDENZE E IMPORT
# Eseguire una volta all'avvio del runtime Google Colab
# ============================================================

# Dipendenze di sistema richieste da Marker PDF
!apt-get update -qq
!apt-get install -y -qq libvips zstd

# Dipendenze Python
%pip install -qU \
    requests \
    beautifulsoup4 \
    pypdf \
    spacy \
    sqlalchemy \
    sentence-transformers \
    transformers \
    accelerate \
    "marker-pdf[full]" \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-ollama \
    langchain-huggingface \
    langchain-qdrant \
    langchain-classic \
    qdrant-client

# Modello linguistico italiano utilizzato dallo SpacyTextSplitter
!python -m spacy download it_core_news_sm -q

# Installazione di Ollama, solo se non è già disponibile nel runtime
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh


# ============================================================
# LIBRERIE STANDARD PYTHON
# ============================================================

import hashlib
import json
import math
import os
import re
import subprocess
import time


from typing import Dict
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse


# ============================================================
# CRAWLING ED ESTRAZIONE DEI CONTENUTI
# ============================================================

import requests

from bs4 import BeautifulSoup
from google.colab import drive
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict


# ============================================================
# LANGCHAIN: DOCUMENTI, CHUNKING E MODELLI
# ============================================================

from langchain_classic.indexes import SQLRecordManager, index
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    SpacyTextSplitter,
)
from collections import deque


# ============================================================
# QDRANT
# ============================================================

from qdrant_client import QdrantClient, models

print("✅ Installazione completata e librerie importate correttamente.")

# Crawling e chunking preliminare del sito





Questa sezione contiene la prima versione della pipeline utilizzata per
l'acquisizione dei contenuti del sito DIEM.

A partire dall'URL iniziale di una sezione, il crawler esplora le pagine
interne fino alla profondità indicata, estrae il testo delle pagine HTML
e dei documenti PDF e applica un primo chunking mediante
`SpacyTextSplitter`.

Per ogni pagina vengono salvati in un file JSON:

- i chunk preliminari;
- l'URL sorgente;
- l'hash del contenuto;
- la profondità raggiunta durante il crawling;
- il tipo di documento;
- la data di estrazione.

Questi chunk rappresentano una prima elaborazione grezza dei contenuti.
Nella fase successiva, gli URL memorizzati nei JSON vengono utilizzati
per recuperare nuovamente le pagine e applicare un chunking più avanzato,
insieme all'estrazione dei metadati specifici per ciascuna sezione.

In [ ]:
# --- CONFIGURAZIONE BLACKLIST ---
# Inserisci qui porzioni di URL o domini da evitare
# come aree private, login, sitemap o pagine in altre lingue.

BLACKLIST_URLS = [
    "sitemap",
    "/en/",
    "/english",
    "?language=en",
    "lang=en"
]


def calcola_hash(testo):
    return hashlib.sha256(testo.encode("utf-8")).hexdigest()


def estrai_testo_AVANZATO(html_content):
    """Logica di pulizia specifica per Unisa applicata al contenuto HTML."""

    soup = BeautifulSoup(html_content, "html.parser")

    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "aside",
        "iframe",
        "head",
        "noscript"
    ]):
        tag.decompose()

    # Elimina gli elementi nascosti tramite stile inline.
    for tag in soup.find_all(
        style=re.compile(r"display:\s*none")
    ):
        tag.decompose()

    classi_da_eliminare = [
        ".breadcrumb",
        ".col-md-6",
        ".col-md-3",
        ".btn",
        ".btn-primary",
        ".btn.btn-primary",
        ".bg-footer",
        ".sub-footer",
        ".hidden",
        ".dropdown-menu",
        ".collapse.navbar-collapse",
        ".next",
        ".prev",
        ".close",
        ".sr-only",
        ".off-bar.collapse"
    ]

    for s in classi_da_eliminare:
        for e in soup.select(s):
            e.decompose()

    # Estrazione e pulizia degli spazi.
    text = soup.get_text(separator=" ")

    # Pulizia delle stringhe residue.
    blacklist = [
        "skip to main content",
        "vai al contenuto",
        "stampa la pagina",
        "condividi"
    ]

    for frase in blacklist:
        text = re.compile(
            re.escape(frase),
            re.IGNORECASE
        ).sub("", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


# --- CRAWLER CON LANGCHAIN ---

def crawl_unisa_langchain(
    start_url,
    max_depth,
    nome,
    chunk_size=1000,
    chunk_overlap=100
):
    base_path = (
        f"/content/drive/Shareddrives/LLM/home/{nome}/"
    )

    os.makedirs(base_path, exist_ok=True)

    # Inizializzazione dello splitter di LangChain.
    text_splitter = SpacyTextSplitter(
        pipeline="it_core_news_sm",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    visitati = set()
    hash_visitati = set()
    risultati_finali = []

    start_url = (
        start_url
        .replace("http://", "https://")
        .rstrip("/")
    )

    coda = [(start_url, 0)]
    domain = urlparse(start_url).netloc

    print(
        f"🚀 Start LangChain Crawler: {start_url} "
        f"(Max Depth: {max_depth})"
    )

    parsed_start = urlparse(start_url)
    domain_root = (
        f"{parsed_start.scheme}://{parsed_start.netloc}"
    )

    while coda:
        url, depth = coda.pop(0)

        # Controllo blacklist e duplicati.
        if (
            url in visitati
            or depth > max_depth
            or any(
                black in url
                for black in BLACKLIST_URLS
            )
        ):
            continue

        visitati.add(url)

        print(f"[{depth}] Processing: {url}")

        try:
            # 1. GESTIONE PDF
            if url.lower().endswith(".pdf"):
                loader = PyPDFLoader(url)

                # Il loader restituisce un documento per pagina.
                docs = loader.load()

                testo_completo = " ".join([
                    d.page_content
                    for d in docs
                ])

                chunks = text_splitter.split_text(
                    testo_completo
                )

            # 2. GESTIONE HTML
            else:
                risposta = requests.get(url)
                risposta.raise_for_status()

                html_puro = risposta.text

                testo_completo = estrai_testo_AVANZATO(
                    html_puro
                )

                chunks = text_splitter.split_text(
                    testo_completo
                )

                # Navigazione ricorsiva BFS limitata da max_depth.
                if depth < max_depth:
                    soup = BeautifulSoup(
                        html_puro,
                        "html.parser"
                    )

                    for a in soup.find_all(
                        "a",
                        href=True
                    ):
                        raw_link = urljoin(
                            url,
                            a["href"]
                        ).split("#")[0]

                        if "/home/uploads/" in raw_link:
                            raw_link = raw_link.replace(
                                "/home/uploads/",
                                "/uploads/"
                            )

                        full_link = raw_link

                        full_link = (
                            full_link
                            .replace("http://", "https://")
                            .rstrip("/")
                        )

                        # Accetta:
                        # A. pagine appartenenti alla sezione iniziale;
                        # B. documenti PDF collegati alla sezione.
                        is_valid_path = (
                            full_link.startswith(start_url)
                            or full_link.lower().endswith(".pdf")
                        )

                        if not is_valid_path:
                            continue

                        # Controllo del dominio e degli URL visitati.
                        if (
                            urlparse(full_link).netloc == domain
                            and full_link not in visitati
                        ):
                            if depth < max_depth:
                                coda.append(
                                    (full_link, depth + 1)
                                )

            # 3. SALVATAGGIO DEI RISULTATI
            content_hash = calcola_hash(
                testo_completo
            )

            # Controllo anti-duplicato sul contenuto.
            if content_hash in hash_visitati:
                print(
                    f"🔄 Duplicato di contenuto "
                    f"ignorato: {url}"
                )
                continue

            hash_visitati.add(content_hash)

            risultati_finali.append({
                "chunks": chunks,
                "metadata": {
                    "source": url,
                    "hash": content_hash,
                    "depth": depth,
                    "type": (
                        "pdf"
                        if url.endswith(".pdf")
                        else "html"
                    ),
                    "extracted_at": time.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    )
                }
            })

        except Exception as e:
            print(f"❌ Errore su {url}: {e}")

    # Output JSON su Drive.
    output_file = (
        f"{base_path}/"
        f"data_{nome}_{int(time.time())}.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            risultati_finali,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(
        f"\n✅ Fine. {len(risultati_finali)} "
        f"pagine salvate in {output_file}"
    )

# # --- UTILIZZO ---
# crawl_unisa_langchain(
#     start_url="https://www.diem.unisa.it/home/bandi",
#     max_depth=4,
#     nome = "bandi"
# )

# Chunking avanzato

Il primo crawling del sito ha portato al salvataggio di numerosi file.json, con ogni campo source visitato salvato.
Con il chunking avanzato non si è diviso più solo in base all'html, ma si è effettuata conversione in markdown e suddivisione intelligente con Spacy Text Splitter per il testo e Recursive Text Splitter per le tabelle.

Inoltre l'utilizzo del modello LLM Gamma 2 ha permesso di ottenere dei metadati associati ad ogni chunk, incluse delle questions utili per una ricerca semantica basata su di esse.

Funzioni utili

In [ ]:
def estrai_testo_STRUTTURATO(html_content):
    soup = BeautifulSoup(html_content, "html.parser")

    # 1. RIMOZIONE ELEMENTI INUTILI (BOILERPLATE)
    # Rimuoviamo i tag che non contengono mai informazioni utili per il RAG
    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "aside",
        "iframe",
        "head",
        "noscript",
        "svg"
    ]):
        tag.decompose()

    # Rimuoviamo le classi specifiche identificate nel sito UNISA
    classi_da_eliminare = [
        ".breadcrumb",
        ".col-md-6",
        ".col-md-3",
        ".btn",
        ".btn-primary",
        ".btn.btn-primary",
        ".bg-footer",
        ".sub-footer",
        ".hidden",
        ".dropdown-menu",
        ".collapse.navbar-collapse",
        ".next",
        ".prev",
        ".close",
        ".sr-only",
        ".off-bar.collapse",
        "#skiptocontent"
    ]

    for selettore in classi_da_eliminare:
        for elemento in soup.select(selettore):
            elemento.decompose()

    # 2. PRESERVAZIONE DELLA STRUTTURA
    # Gestione titoli: trasformiamo h1, h2, h3 e h4 in Markdown
    for i in range(1, 5):
        for h in soup.find_all(f"h{i}"):
            testo = h.get_text(strip=True)

            if testo:
                h.replace_with(
                    f"\n\n{'#' * i} {testo}\n"
                )

    # Gestione tabelle: trasformazione in tabella Markdown
    for table in soup.find_all("table"):
        rows = table.find_all("tr")
        markdown_table = ["\n"]

        for row in rows:
            cells = [
                cell.get_text(strip=True)
                for cell in row.find_all(["th", "td"])
            ]

            if cells:
                markdown_table.append(
                    "| " + " | ".join(cells) + " |"
                )

        markdown_table.append("\n")

        table.replace_with(
            "\n".join(markdown_table)
        )

    # Gestione liste: trasformazione in bullet point
    for li in soup.find_all("li"):
        testo_li = li.get_text(strip=True)

        if testo_li:
            li.replace_with(
                f"\n* {testo_li}"
            )

    # 3. ESTRAZIONE E PULIZIA FINALE
    # Usiamo \n come separatore per mantenere le divisioni create
    text = soup.get_text(separator="\n")

    # Pulizia degli spazi bianchi eccessivi
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


# Caricamento dei modelli di Marker
# L'operazione viene eseguita una sola volta, fuori dalle funzioni.
print("🧠 Caricamento modelli AI di Marker in corso...")

converter = PdfConverter(
    artifact_dict=create_model_dict()
)


def estrai_testo_PDF_STRUTTURATO(pdf_url):
    """
    Scarica un PDF e lo converte in Markdown utilizzando Marker.
    """

    temp_pdf = "temp_test.pdf"

    try:
        # 1. Download del PDF
        print(
            f"📥 Scaricamento PDF da: {pdf_url}..."
        )

        response = requests.get(
            pdf_url,
            timeout=30
        )

        response.raise_for_status()

        with open(temp_pdf, "wb") as f:
            f.write(response.content)

        # 2. Conversione del PDF in Markdown
        print(
            "⚡ Conversione in corso "
            "(AI Layout Analysis)..."
        )

        result = converter(temp_pdf)
        full_text = result.markdown

        # 3. Rimozione del file temporaneo
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)

        return full_text

    except Exception as e:
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)

        raise e


# Avvio del server Ollama
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Server pronto!")


# Download del modello utilizzato per l'estrazione dei metadati
!ollama pull qwen3:14b


def ensure_ollama_running():
    print("Checking Ollama server status...")

    # Verifica se il server Ollama è già attivo
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            check=False,
            timeout=5
        )

        if (
            result.returncode == 0
            and "NAME" in result.stdout
        ):
            print(
                "Ollama server is already running."
            )
            return

    except (
        subprocess.CalledProcessError,
        FileNotFoundError,
        subprocess.TimeoutExpired
    ):
        pass

    print(
        "Ollama server not found or not responding. "
        "Attempting to restart..."
    )

    # Arresta eventuali processi Ollama non responsivi
    subprocess.run(
        ["pkill", "-f", "ollama serve"],
        check=False
    )

    time.sleep(1)

    # Riavvia Ollama in background
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(5)

    # Verifica l'avvio del server
    for i in range(10):
        try:
            result = subprocess.run(
                ["ollama", "list"],
                capture_output=True,
                text=True,
                check=False,
                timeout=5
            )

            if (
                result.returncode == 0
                and "NAME" in result.stdout
            ):
                print(
                    "Ollama server started "
                    "and is responsive."
                )
                return

        except (
            subprocess.CalledProcessError,
            FileNotFoundError,
            subprocess.TimeoutExpired
        ):
            pass

        print(
            "Ollama not yet responsive, "
            f"retrying... ({i + 1}/10)"
        )

        time.sleep(3)

    raise RuntimeError(
        "Failed to start Ollama server "
        "after multiple retries."
    )

In [ ]:
def calcola_hash(testo):
    return hashlib.sha256(testo.encode('utf-8')).hexdigest()


# 1. SETUP AMBIENTE
drive.mount("/content/drive")


# Configurazione Ollama
llm = ChatOllama(
    model="qwen3:14b",
    temperature=0.0,
    format="json"
)


# Configurazione dello splitter Markdown
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)


def is_valid_chunk(text):
    # Rimuove gli spazi e i simboli Markdown comuni per il conteggio
    clean_text = re.sub(
        r"[#\-\*\_ \n]",
        "",
        text
    )

    # Se dopo la pulizia rimangono meno di 10 caratteri,
    # il chunk è quasi certamente rumore
    if len(clean_text) < 10:
        return False

    # Esclude alcuni contenuti non informativi
    if text.strip() in [
        "-",
        "####",
        "###",
        "#### -",
        "Torna su"
    ]:
        return False

    return True


def ottieni_sotto_chunks(
    testo_chunk,
    target_size_testo=2000,
    target_size_tabella=6000
):
    lunghezza = len(testo_chunk)

    # 1. Identificazione del tipo di contenuto
    is_tabella = (
        "|" in testo_chunk
        and "---" in testo_chunk
    )

    max_chars = (
        target_size_tabella
        if is_tabella
        else target_size_testo
    )

    # Se il chunk è già piccolo, non fare nulla
    if lunghezza <= max_chars:
        return [testo_chunk]

    # 2. Calcolo bilanciato
    num_parti = math.ceil(
        lunghezza / max_chars
    )

    dimensione_bilanciata = math.ceil(
        lunghezza / num_parti
    )

    print(
        f"   ⚖️ Bilanciamento "
        f"({'Tabella' if is_tabella else 'Testo'}): "
        f"{lunghezza} caratteri in {num_parti} parti "
        f"da ~{dimensione_bilanciata}"
    )

    # 3. Scelta dello splitter in base al tipo
    if is_tabella:
        # Per le tabelle usiamo Recursive per preservare i \n
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=6000,
            chunk_overlap=200,
            separators=[
                "\n\n",
                "\n",
                " ",
                ""
            ]
        )

    else:
        # Per il testo usiamo Spacy per la coerenza linguistica
        splitter = SpacyTextSplitter(
            pipeline="it_core_news_sm",
            chunk_size=dimensione_bilanciata,
            chunk_overlap=200
        )

    return splitter.split_text(
        testo_chunk
    )

## Estrazione dei metadati specifici per sezione

Dopo il recupero degli URL e la creazione dei chunk strutturati, ogni sezione
del sito è stata elaborata mediante la stessa pipeline generale:

1. recupero del contenuto HTML o PDF;
2. conversione in testo strutturato;
3. suddivisione in chunk;
4. estrazione dei metadati tramite Gemma 2;
5. validazione ed eventuale correzione del JSON generato;
6. creazione dei documenti LangChain;
7. salvataggio dei risultati e del manifest dei metadati.

Il prompt e lo schema dei metadati sono stati adattati alla natura dei
contenuti presenti nelle diverse sezioni. Ad esempio:

- i profili dei docenti includono nome, cognome e insegnamenti;
- eventi e news includono data e tipologia del contenuto;
- i contatti includono ruolo, struttura e recapiti;
- la mobilità internazionale include programma, destinatari, paese,
  università partner e scadenze;
- i bandi includono anno, stato, tipologia, destinatari e scadenza.

Di seguito viene riportata, come esempio rappresentativo, la pipeline
utilizzata per la sezione Bandi. Le altre sezioni utilizzavano la stessa
struttura, modificando il prompt e i metadati specifici da estrarre.

## Per i bandi

In [ ]:
def correggi_json(modello, chunk_originale, json_rotto, errore):
    """
    Invia il chunk originale e il JSON malformato per una riparazione accurata.
    """
    recover_prompt = f"""
    Il JSON che hai generato è incompleto o sintatticamente errato.
    Errore rilevato: {errore}

    JSON DA RIPARARE:
    {json_rotto}

    TESTO ORIGINALE DI RIFERIMENTO (Usa questo per completare le informazioni):
    {chunk_originale}

    ATTIVITÀ:
    1. Sei un estrattore di dati puro. Non riassumere, non parafrasare e non commentare il testo.
       Il tuo unico output deve essere l'oggetto JSON richiesto descritto di seguito
    2. Completa il JSON basandoti ESCLUSIVAMENTE sul testo originale.
    3. Se il testo era troncato, usa il riferimento per compilare correttamente i campi.
    4. Rispondi ESCLUSIVAMENTE con il JSON corretto.
    5. Restituisci ESCLUSIVAMENTE un oggetto JSON con questa struttura:
    {{
      "keywords": ["key1", "key2"],
      "questions": ["domanda1", "domanda2"],
      "scadenza": "GG/MM/AAAA o 'N.D.' se non trovata",
      "anno": "anno di riferimento",
      "stato": "Aperto / Scaduto / In Corso / Graduatoria Pubblicata",
      "tipologia": "Assegno di Ricerca / Borsa / Tutorato / Incarico Docenza / ecc.",
      "target": "Studenti / Laureati / Dottori di Ricerca",
      "dipartimento": "Nome del Dipartimento che pubblica il bando (es. DIEM, Medicina, ecc.)"
    }}
    6. Assicurati che il JSON creato sia sintatticamente corretto.
    dove questions sono domande d'esempio a cui il testo risponde.
    """
    print("   RECOVER: Riparazione basata sul contesto originale...")
    return modello.invoke(recover_prompt).content.strip()


def aggiungi_al_manifest(set_destinazione, dato):
    """
    Aggiunge in modo sicuro stringhe o liste a un set di metadati,
    evitando l'errore 'unhashable type: list'.
    """
    if dato is None:
        return

    if isinstance(dato, list):
        # Se è una lista, puliamo e aggiungiamo gli elementi uno per uno
        for elemento in dato:
            if elemento and isinstance(elemento, str):
                set_destinazione.add(elemento.strip())
            elif elemento and not isinstance(elemento, list):
                # Evita liste di liste
                set_destinazione.add(str(elemento).strip())

    elif isinstance(dato, str):
        if dato.strip():
            set_destinazione.add(dato.strip())

    else:
        # Se è un altro tipo (es. int), lo trasformiamo in stringa
        set_destinazione.add(str(dato).strip())


def processa_cartella_json(cartella_input, cartella_output):
    """
    Scansiona una cartella, legge tutti i file JSON e processa ogni URL contenuto.
    """
    # Creazione cartella di output se non esiste
    if not os.path.exists(cartella_output):
        os.makedirs(cartella_output)
        print(f"📁 Creata cartella di output: {cartella_output}")

    # Lista tutti i file .json nella cartella
    files_json = [
        f for f in os.listdir(cartella_input)
        if f.endswith(".json")
    ]

    if not files_json:
        print(f"⚠️ Nessun file JSON trovato in {cartella_input}")
        return

    print(f"📚 Trovati {len(files_json)} file da processare.")

    manifest_metadati = {
        "cartella_sorgente": cartella_input,
        "valori_filtro_unici": {
            "sections": set(),
            "subsections": set(),
            "doc_types": set(),
            "sources": set(),
            "keywords": set(),
            "questions": set(),
            "scadenze": set(),
            "anni": set(),
            "stati": set(),
            "tipologie": set(),
            "target": set(),
            "dipartimenti": set()
        }
    }

    for nome_file in files_json:
        path_input_file = os.path.join(
            cartella_input,
            nome_file
        )

        path_output_file = os.path.join(
            cartella_output,
            nome_file
        )

        print(f"\n🚀 Inizio elaborazione file: {nome_file}")

        with open(
            path_input_file,
            "r",
            encoding="utf-8"
        ) as f:
            try:
                data = json.load(f)

            except Exception as e:
                print(f"❌ Errore lettura {nome_file}: {e}")
                continue

        risultati_file = []

        # Processamento degli URL nel file corrente
        for entry in data:
            metadata_sorgente = entry.get(
                "metadata",
                {}
            )

            url = metadata_sorgente.get(
                "source"
            )

            doc_type = metadata_sorgente.get(
                "type",
                ""
            ).lower()

            if not url:
                continue

            print(f"  🌐 Ingestione: {url}...")

            try:
                # 1. Estrazione Markdown
                # Usa le funzioni definite in precedenza
                if doc_type == "pdf":
                    testo_markdown = (
                        estrai_testo_PDF_STRUTTURATO(url)
                    )

                else:
                    resp = requests.get(
                        url,
                        timeout=15
                    )

                    resp.raise_for_status()

                    testo_markdown = (
                        estrai_testo_STRUTTURATO(
                            resp.text
                        )
                    )

                hash = calcola_hash(
                    testo_markdown
                )

                # 2. Splitting
                segments = markdown_splitter.split_text(
                    testo_markdown
                )

                # 3. Riformulazione LLM
                for seg in segments:
                    if not is_valid_chunk(
                        seg.page_content
                    ):
                        print(
                            "  ⏩ Chunk saltato "
                            "(Rumore rilevato): "
                            f"{seg.page_content[:30]}"
                        )
                        continue

                    sotto_chunks = ottieni_sotto_chunks(
                        seg.page_content
                    )

                    for sub_content in sotto_chunks:
                        prompt = f"""
                        Sei l'assistente ufficiale del Dipartimento DIEM dell'Università di Salerno.
                        Il tuo compito è estrarre informazioni chiave da chunk di testo che riguardano bandi pubblicati da Unisa.
                        Sei un estrattore di dati puro. Non riassumere, non parafrasare e non commentare il testo.
                        Il tuo unico output deve essere l'oggetto JSON richiesto descritto di seguito nelle regole.

                        Di seguito le tue attività e le regole da rispettare:

                        ATTIVITÀ:
                        1. Estrai 5-8 parole chiave tecniche (inclusi anni di riferimento, es. "2025").
                        2. Formula 2-3 domande d'esempio a cui il testo risponde.
                        3. Identifica la scadenza, l'anno di riferimento, lo stato, la tipologia, il target e il dipartimento.
                        Come dipartimento devi specificare il dipartimento che pubblica il bando, non necessariamente il DIEM.

                        REGOLE:
                        1. Non utilizzare la tua conoscenza generale per inventare informazioni.
                        2. Se il testo non è utile, rispondi: INVALIDO.
                        3. Restituisci ESCLUSIVAMENTE un oggetto JSON con questa struttura:
                        {{
                          "keywords": ["key1", "key2"],
                          "questions": ["domanda1", "domanda2"],
                          "scadenza": "GG/MM/AAAA o 'N.D.' se non trovata",
                          "anno": "anno di riferimento",
                          "stato": "Aperto / Scaduto / In Corso / Graduatoria Pubblicata",
                          "tipologia": "Assegno di Ricerca / Borsa / Tutorato / Incarico Docenza / ecc.",
                          "target": "Studenti / Laureati / Dottori di Ricerca",
                          "dipartimento": "Nome del Dipartimento (es. DIEM, Medicina, ecc.)"
                        }}
                        4. Assicurati che il JSON creato sia sintatticamente corretto.

                        Contesto (Metadata): {seg.metadata}
                        TESTO DA ELABORARE:
                        {sub_content}
                        """

                        ensure_ollama_running()

                        max_retries = 1
                        tentativo = 0
                        successo = False

                        response = llm.invoke(
                            prompt
                        ).content.strip()

                        while (
                            tentativo <= max_retries
                            and not successo
                        ):
                            if (
                                response.upper()
                                != "INVALIDO"
                            ):
                                try:
                                    # Cerchiamo tutto ciò che sta
                                    # tra la prima { e l'ultima }
                                    match = re.search(
                                        r"(\{.*\})",
                                        response,
                                        re.DOTALL
                                    )

                                    if match:
                                        json_str = (
                                            match.group(1)
                                            .strip()
                                        )

                                        ai_data = json.loads(
                                            json_str,
                                            strict=False
                                        )

                                        valori_filtro = (
                                            manifest_metadati[
                                                "valori_filtro_unici"
                                            ]
                                        )

                                        valori_filtro[
                                            "sections"
                                        ].add(
                                            seg.metadata.get(
                                                "Header 1",
                                                ""
                                            )
                                        )

                                        valori_filtro[
                                            "subsections"
                                        ].add(
                                            seg.metadata.get(
                                                "Header 2",
                                                ""
                                            )
                                        )

                                        valori_filtro[
                                            "doc_types"
                                        ].add(
                                            doc_type
                                        )

                                        valori_filtro[
                                            "sources"
                                        ].add(
                                            url
                                        )

                                        # Metadati generati dall'AI
                                        # potenzialmente instabili come tipo
                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "keywords"
                                            ],
                                            ai_data.get(
                                                "keywords",
                                                []
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "questions"
                                            ],
                                            ai_data.get(
                                                "questions",
                                                []
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "scadenze"
                                            ],
                                            ai_data.get(
                                                "scadenza",
                                                ""
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "anni"
                                            ],
                                            ai_data.get(
                                                "anno",
                                                ""
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "stati"
                                            ],
                                            ai_data.get(
                                                "stato",
                                                ""
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "tipologie"
                                            ],
                                            ai_data.get(
                                                "tipologia",
                                                ""
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "target"
                                            ],
                                            ai_data.get(
                                                "target",
                                                ""
                                            )
                                        )

                                        aggiungi_al_manifest(
                                            valori_filtro[
                                                "dipartimenti"
                                            ],
                                            ai_data.get(
                                                "dipartimento",
                                                ""
                                            )
                                        )

                                        if len(sub_content) == 0:
                                            continue

                                        risultati_file.append({
                                            "page_content": (
                                                sub_content
                                            ),
                                            "metadata": {
                                                "source": url,
                                                "hash": hash,
                                                "extracted_at": (
                                                    time.strftime(
                                                        "%Y-%m-%d "
                                                        "%H:%M:%S"
                                                    )
                                                ),
                                                "doc_type": (
                                                    doc_type
                                                ),
                                                "section": (
                                                    seg.metadata.get(
                                                        "Header 1",
                                                        ""
                                                    )
                                                ),
                                                "subsection": (
                                                    seg.metadata.get(
                                                        "Header 2",
                                                        ""
                                                    )
                                                ),
                                                "keywords": (
                                                    ai_data.get(
                                                        "keywords",
                                                        []
                                                    )
                                                ),
                                                "questions": (
                                                    ai_data.get(
                                                        "questions",
                                                        []
                                                    )
                                                ),
                                                "scadenza": (
                                                    ai_data.get(
                                                        "scadenza",
                                                        ""
                                                    )
                                                ),
                                                "anno": (
                                                    ai_data.get(
                                                        "anno",
                                                        ""
                                                    )
                                                ),
                                                "stato": (
                                                    ai_data.get(
                                                        "stato",
                                                        ""
                                                    )
                                                ),
                                                "tipologia": (
                                                    ai_data.get(
                                                        "tipologia",
                                                        ""
                                                    )
                                                ),
                                                "target": (
                                                    ai_data.get(
                                                        "target",
                                                        ""
                                                    )
                                                ),
                                                "dipartimento": (
                                                    ai_data.get(
                                                        "dipartimento",
                                                        ""
                                                    )
                                                ),
                                                "all_headers": (
                                                    str(
                                                        seg.metadata
                                                    )
                                                )
                                            }
                                        })

                                        successo = True

                                    else:
                                        print(
                                            "  ⚠️ Nessun JSON "
                                            "trovato nella risposta "
                                            f"per {url}"
                                        )

                                        print(response)

                                        raise ValueError(
                                            "Nesssun blocco json "
                                            "trovato"
                                        )

                                except Exception as e:
                                    tentativo += 1

                                    if (
                                        tentativo
                                        <= max_retries
                                    ):
                                        print(
                                            "   ⚠️ Errore JSON "
                                            "al tentativo "
                                            f"{tentativo}: {e}. "
                                            "Provo il recupero..."
                                        )

                                        response = correggi_json(
                                            llm,
                                            sub_content,
                                            response,
                                            str(e)
                                        )

                                    else:
                                        print(
                                            "   ❌ Fallito recupero "
                                            f"dopo {max_retries} "
                                            f"tentativi per {url}"
                                        )

            except Exception as e:
                print(f"  ❌ Errore su {url}: {e}")

        cartella_padre = os.path.dirname(
            path_output_file
        )

        if not os.path.exists(
            cartella_padre
        ):
            os.makedirs(
                cartella_padre,
                exist_ok=True
            )

            print(
                "📁 Creata cartella mancante "
                f"(last minute): {cartella_padre}"
            )

        # Scrittura del file elaborato
        try:
            with open(
                path_output_file,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    risultati_file,
                    f,
                    indent=4,
                    ensure_ascii=False
                )

            print(
                "✅ Salvato correttamente: "
                f"{path_output_file}"
            )

        except Exception as e:
            print(
                "❌ Errore fatale durante "
                f"la scrittura del file: {e}"
            )

    manifest_finalizzato = {}

    for k, v in manifest_metadati[
        "valori_filtro_unici"
    ].items():
        if isinstance(v, set):
            manifest_finalizzato[k] = list(v)
        else:
            manifest_finalizzato[k] = v

    manifest_metadati[
        "valori_filtro_unici"
    ] = manifest_finalizzato

    path_manifest = os.path.join(
        cartella_output,
        "metadata_manifest.json"
    )

    with open(
        path_manifest,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest_metadati,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"✅ Metadati salvati: {path_manifest}"
    )


# --- ESECUZIONE DI ESEMPIO ---

nome = "/home/bandi"

CARTELLA_SORGENTE = (
    f"/content/drive/Shareddrives/LLM/{nome}"
)

CARTELLA_DESTINAZIONE = (
    "/content/drive/Shareddrives/LLM/"
    f"chunks_aggiornati/{nome}"
)

processa_cartella_json(
    CARTELLA_SORGENTE,
    CARTELLA_DESTINAZIONE
)

#Popolamento e indicizzazione in Qdrant



I chunk elaborati vengono caricati in collezioni Qdrant separate in base
alla tipologia di contenuto. Per ogni chunk viene memorizzato il vettore
del testo e, quando disponibili, due vettori aggiuntivi associati alle
domande generate durante l'estrazione dei metadati.

L'indicizzazione incrementale utilizza `SQLRecordManager` per riconoscere
i documenti già elaborati e aggiornare soltanto i contenuti modificati.


In [ ]:
# 2. Definisci il modello di Embedding
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 16}
)

#3. Mappa delle collezioni e relativi percorsi
COLLECTIONS_MAP = {
    "docenti": [
        "/content/drive/Shareddrives/LLM/chunks_pro/docenti",
        "/content/drive/Shareddrives/LLM/chunks_pro/home/contatti"
    ],
    "international": [
        "/content/drive/Shareddrives/LLM/chunks_pro/international"
    ],
    "bandi": [
        "/content/drive/Shareddrives/LLM/chunks_pro/home/bandi"
    ],
    "dipartimento": [
        "/content/drive/Shareddrives/LLM/chunks_pro/dipartimento",
        "/content/drive/Shareddrives/LLM/chunks_pro/consiglio_didattico"
    ],
    "offerta_formativa":[
        "/content/drive/Shareddrives/LLM/chunks_pro/offerta_formativa/Triennale",
        "/content/drive/Shareddrives/LLM/chunks_pro/offerta_formativa/Magistrale",
        "/content/drive/Shareddrives/LLM/chunks_pro/offerta_formativa/Dottorato"
    ],
    "iniziative_ricerca_alternanza": [
        "/content/drive/Shareddrives/LLM/chunks_pro/home/news",
        "/content/drive/Shareddrives/LLM/chunks_pro/home/eventi",
        "/content/drive/Shareddrives/LLM/chunks_pro/home/TerzaMissione",
        "/content/drive/Shareddrives/LLM/chunks_pro/ricerca",
        "/content/drive/Shareddrives/LLM/chunks_pro/alternanza_scuola_lavoro",
        "/content/drive/Shareddrives/LLM/chunks_pro/focus_didattica"
    ]
}





def pulisci_e_valida_chunk(content):
    """
    Pulisce il testo dalle immagini e valida il contenuto proteggendo le tabelle.
    """
    # 1. PULIZIA: Rimuoviamo solo i pattern spazzatura (immagini)
    testo_pulito = re.sub(r'!\[.*\]|!page_image|\[image\]', '', content, flags=re.IGNORECASE)
    testo_pulito = testo_pulito.strip()

    # 2. RICONOSCIMENTO TABELLE:
    # Cerchiamo pattern tipici delle tabelle (almeno due righe con separatori | o sequenze di spazi)
    # Una tabella markdown ha solitamente righe che iniziano/finiscono con | o hanno | all'interno
    is_table = len(re.findall(r'\|', testo_pulito)) > 4 or len(re.findall(r'\t', testo_pulito)) > 2

    # 3. CRITERI DI VALIDAZIONE

    # Se è una tabella, la teniamo quasi sempre (abbassiamo la soglia di lunghezza)
    if is_table:
        # Contiamo i caratteri alfanumerici reali presenti nella tabella
        testo_reale_tabella = re.findall(r'\w', testo_pulito)

        # Una tabella valida deve avere almeno un minimo di testo (es. almeno 10 lettere/numeri)
        if len(testo_reale_tabella) > 15:
            return content, True
        else:
            return None, False # Se ha solo trattini e spazi, viene eliminata!


    # Se NON è una tabella, applichiamo i filtri standard per il testo narrativo
    else:

        # Calcoliamo anche il numero di parole
        numero_parole = len(testo_pulito.split())

        # SOGLIA RICALIBRATA: Scarta se troppo corto in caratteri O se ha pochissime parole
        if len(testo_pulito) < 85 or numero_parole < 12:
            return None, False


        # Rapporto lettere/simboli per testo normale
        # Cerca qualsiasi carattere che sia considerato una "lettera" in Unicode
        letters = re.findall(r'\w', testo_pulito)
        # Nota: \w include anche i numeri e l'underscore.
        if len(testo_pulito) > 0 and len(letters) / len(testo_pulito) < 0.25:
            return None, False

    # Se il chunk ha superato i controlli o è una tabella valida
    return content, True


# Funzione helper per leggere JSON e convertirli in oggetti Document di LangChain
def load_docs_from_paths(paths):
    docs = []

    def process_file(file_path, cartella_origine):
        if not file_path.endswith('.json'):
            return
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

                # Logica retrieval precedente e successivo
                # Questo dizionario terrà traccia del chunk_index corrente per OGNI url.
                # Struttura: {"url_1": 0, "url_2": 0, ...}
                url_counters = {}

                for item in data:
                    # 1. Recupera il contenuto testuale (priorità a page_content)
                    content = item.get('page_content')
                    content, flag = pulisci_e_valida_chunk(content)

                    # 2. Se abbiamo page_content, procediamo con il nuovo formato
                    if content and isinstance(content, str) and content.strip():
                        raw_meta = item.get('metadata', {})

                        # Aggiunta metadato per refresh del db
                        raw_meta['cartella_origine'] = cartella_origine

                        # Recuperiamo l'URL di questo specifico chunk
                        current_source = raw_meta.get('source')

                        # Se l'URL non è mai stato visto prima in questo file,
                        # inizializziamo il suo contatore a 0 (Ricomincia da 0!)
                        if current_source not in url_counters:
                            url_counters[current_source] = 0

                        # Assegniamo l'indice corrente isolato per questo URL
                        raw_meta['chunk_index'] = url_counters[current_source]

                        # Incrementiamo il contatore SOLO per questo specifico URL per il prossimo chunk
                        url_counters[current_source] += 1

                        docs.append(Document(page_content=content, metadata=raw_meta))

        except Exception as e:
            print(f"Errore nella lettura di {file_path}: {e}")

    for path in paths:
        cartella_origine = os.path.basename(path.strip('/'))

        if os.path.isdir(path):
            for root, _, files in os.walk(path):
                for file in files:
                    process_file(os.path.join(root, file), cartella_origine)
        elif os.path.isfile(path):
            process_file(path, cartella_origine)

    return docs

# 4. Inizializzazione DB e indicizzazione
# Scegliamo una cartella dove salvare i file di Chroma e del Record Manager SQLite
PERSIST_DIRECTORY = "/content/drive/Shareddrives/LLM/qdrant_db_pro_questionsdistinte"
RECORD_MANAGER_DB_URL = f"sqlite:///{PERSIST_DIRECTORY}/record_manager_cache.sql"
# qui si salverà sul drive una istanza dell'sql. Gli hash degli url sono salvati e aggiornati sul drive. Non dobbiamo ricalcolarli ogni volta

os.makedirs(PERSIST_DIRECTORY, exist_ok=True)

# Inizializziamo il client nativo di Qdrant che salva su disco (Drive)
# 1. Chiude eventuali istanze precedenti rimaste aperte nella memoria di Colab
if 'client' in globals():
    try:
        client.close()
        print("Istanza Qdrant precedente chiusa con successo.")
    except Exception as e:
        pass

# 2. Inizializziamo il client nativo di Qdrant che salva su disco (Drive)
client = QdrantClient(path=PERSIST_DIRECTORY)

for collection_name, paths in COLLECTIONS_MAP.items():
    print(f"\n--- Processando la collezione: {collection_name} ---")

    # Carica i documenti
    docs = load_docs_from_paths(paths)
    print(f"Trovati {len(docs)} chunk da indicizzare.")

    if not docs:
        print("Nessun documento trovato, salto la collezione.")
        continue

    # Il modello BAAI/bge-m3 genera vettori con dimensione 1024
    dimensione_vettore = 1024

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "": models.VectorParams(size=dimensione_vettore, distance=models.Distance.COSINE),
                "vector_question_1": models.VectorParams(size=dimensione_vettore, distance=models.Distance.COSINE),
                "vector_question_2": models.VectorParams(size=dimensione_vettore, distance=models.Distance.COSINE),
            }
        )

    # 3. ISTANZIA QDRANT TRAMITE LANGCHAIN
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )

    # Crea un namespace univoco per il Record Manager basato sul nome della collezione
    namespace = f"qdrant/{collection_name}"
    record_manager = SQLRecordManager(namespace, db_url=RECORD_MANAGER_DB_URL)
    record_manager.create_schema()

    # Esegui l'indicizzazione smart
    # source_id_key="source" usa l'url estratto dai metadati come raggruppatore
    # cleanup="incremental" cancella i vecchi chunk di un URL se il contenuto è cambiato
    indexing_result = index(
        docs,
        record_manager,
        vectorstore,
        cleanup="incremental",
        source_id_key="source"
    )

    print(f"Risultato indicizzazione per {collection_name}: {indexing_result}")

    print(f"Calcolo e inserimento embedding separati per 'questions' in {collection_name}...")

    next_page_offset = None
    totale_aggiornati = 0

    while True:
        punti_qdrant, next_page_offset = client.scroll(
            collection_name=collection_name,
            limit=1000,
            with_payload=True,
            with_vectors=False,
            offset=next_page_offset
        )

        testi_da_embeddare = []
        mappa_vettori = []

        for punto in punti_qdrant:
            payload = punto.payload or {}
            meta = payload.get("metadata", {})
            questions = meta.get("questions", [])

            if isinstance(questions, str):
                questions = [questions]

            if not isinstance(questions, list):
                continue

            questions = [
                domanda.strip()
                for domanda in questions
                if isinstance(domanda, str) and domanda.strip()
            ]

            if not questions:
                continue

            # Se c'e una sola domanda, la usiamo sia per vector_question_1 sia per vector_question_2.
            # Se ce ne sono piu di due, prendiamo solo le prime due.
            if len(questions) == 1:
                questions_da_embeddare = [questions[0], questions[0]]
            else:
                questions_da_embeddare = questions[:2]

            for idx, domanda in enumerate(questions_da_embeddare, start=1):
                testi_da_embeddare.append(domanda)
                mappa_vettori.append((punto.id, f"vector_question_{idx}"))

        if testi_da_embeddare:
            vettori_domande = embeddings.embed_documents(testi_da_embeddare)

            vettori_per_punto = {}

            for (punto_id, nome_vettore), vettore in zip(mappa_vettori, vettori_domande):
                if punto_id not in vettori_per_punto:
                    vettori_per_punto[punto_id] = {}

                vettori_per_punto[punto_id][nome_vettore] = vettore

            points_to_update = [
                models.PointVectors(
                    id=punto_id,
                    vector=vectors
                )
                for punto_id, vectors in vettori_per_punto.items()
            ]

            client.update_vectors(
                collection_name=collection_name,
                points=points_to_update
            )

            totale_aggiornati += len(points_to_update)

        if next_page_offset is None:
            break

    print(f"✅ Aggiornati in totale {totale_aggiornati} chunk con embedding separati per question 1 e question 2.")

    # 6. CREAZIONE INDICI SUI PAYLOAD (METADATI)
    # Una volta creati i documenti, diciamo a Qdrant di indicizzare campi specifici
    # per rendere le ricerche velocissime.
    # 6. CREAZIONE INDICI SUI PAYLOAD (METADATI)
    try:
        # Campi che abbiamo nel JSON ma NON vogliamo indicizzare in Qdrant
        CAMPI_DA_IGNORARE = {"hash", "all_headers", "subsection", "extracted_at", "doc_type",
                             "scadenza", "target", "luogo", "durata", "coordinatore_diem", "requisiti_linguistici", "data_scadenza", "data_stipula",
                             "course_class", "responsible_person", "research_category", "section"}

        # Estrai dinamicamente tutte le chiavi presenti nei metadati dei docs caricati
        chiavi_metadata = set()
        for d in docs:
            chiavi_metadata.update(d.metadata.keys())

        for chiave in chiavi_metadata:
            if chiave in CAMPI_DA_IGNORARE:
                continue # Salta l'indicizzazione per questa chiave

            # Aggiungiamo il prefisso 'metadata.' fondamentale per LangChain
            qdrant_field = f"metadata.{chiave}"

            try:

                # sul campo questions vorrei che "iscrivermi" nella query prenda anche i chunk in cui nelle relative
                # questions compare "iscrivere" o "iscrizione" --> utilizzo lemmatizzazione per essere più flessibile
                # flessibilità maggiore per questions che simula la naturalezza del parlato di un utente.
                if chiave == "questions":
                    client.create_payload_index(
                        collection_name=collection_name,
                        field_name=qdrant_field,
                        field_schema=models.TextIndexParams(
                            type="text",
                            tokenizer=models.TokenizerType.WORD,
                            lowercase=True,

                            stemmer=models.SnowballParams(
                                type=models.Snowball.SNOWBALL,
                                language=models.SnowballLanguage.ITALIAN
                            ),

                            stopwords=models.Language.ITALIAN                       # <--- Attiva le stopwords
                        )
                    )

                # Per gli altri indici testuali invece cerchiamo dei match esatti tra le parole che occorrono
                # nella query e nelle questions.


                # --- INDICE NUMERICO PER I CHUNK ADIACENTI ---
                # Serve al backend, non viene usato per la query e non riguarda l'utente.
                # Permette di ottenere il chunk precedente e quello successivo a quello estratto in tempo O(1)
                elif chiave == "chunk_index":
                    client.create_payload_index(
                        collection_name=collection_name,
                        field_name=qdrant_field,
                        field_schema=models.PayloadSchemaType.INTEGER # <--- Dice a Qdrant che è un numero intero
                    )

                # membri_ruoli, commission, struttura in dipartimento e consiglio didattico
                # tipologia, target e dipartimento in bandi
                # tipologia, nome_progetto, responsabile terza missione
                # titolo news ed eventi
                # ufficio, nominativo in contatti
                # universita_partner, coordinatore_diem in international
                # per offerta formative tutte key, no text
                # responsible_person, research category, project period in ricerca
                elif chiave in [
                                "membri", "commission",
                                "tipologia", "dipartimento",
                                "nome_progetto", "responsabile",
                                "titolo",
                                "ufficio", "nominativo",
                                "universita_partner", "destinatario",
                                "project_period", "programma", "struttura"]:

                    client.create_payload_index(
                        collection_name=collection_name,
                        field_name=qdrant_field,
                        field_schema=models.TextIndexParams(
                            type="text",
                            tokenizer=models.TokenizerType.WORD,
                            lowercase=True,
                        )
                    )
                else:
                    # Indicizza source, keywords, scadenza, ecc. come KEYWORD
                    client.create_payload_index(
                        collection_name=collection_name,
                        field_name=qdrant_field,
                        field_schema=models.PayloadSchemaType.KEYWORD
                    )
            except Exception as e:
                if "already exists" not in str(e).lower():
                    print(f"  ⚠️ Avviso durante la creazione dell'indice per '{qdrant_field}': {e}")

    except Exception as e:
        print(f"❌ Errore generale nella creazione indici: {e}")

print("\nIndicizzazione Qdrant completata!")

# Aggiornamento del vector DB

In [ ]:
# Mappa che associa la collezione alla sua pagina Hub principale
HUBS_MAP = {
    "news": {
        "collezione": "iniziative_ricerca_alternanza",
        "urls": [
            "https://www.diem.unisa.it/home/news" # Sostituisci con gli URL reali delle pagine indice
        ]
    },
    "eventi": {
        "collezione": "iniziative_ricerca_alternanza",
        "urls": [
            "https://www.diem.unisa.it/home/eventi"
        ]
    },
    "bandi": {
        "collezione": "bandi",
        "urls": [

            "https://www.diem.unisa.it/home/bandi?anno=2026&struttura=300638&modulo=139",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=504&struttura=300638",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=316&struttura=300638",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=226",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=67&struttura=300638",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=292&struttura=300638",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=293&cdsStruttura=300638",
            "https://www.diem.unisa.it/home/bandi?anno=2026&modulo=505&struttura=300638"
        ]
    }
}

from bs4 import BeautifulSoup

from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qsl, urlencode

def normalizza_url_unisa(url_grezzo):
    """
    Standardizza l'URL tenendo la whitelist dei parametri ordinati.
    Manteniamo il parametro 'anno' come da tua intuizione.
    """
    parsed = urlparse(url_grezzo)
    query_params = dict(parse_qsl(parsed.query))

    if url_grezzo.lower().endswith('.pdf'):
        return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

    whitelist = {'modulo', 'struttura', 'anno', 'bando', 'idConcorso', 'id'}
    params_puliti = {k: v for k, v in query_params.items() if k in whitelist}

    query_string = urlencode(sorted(params_puliti.items()))
    url_finale = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

    if query_string:
        url_finale += f"?{query_string}"

    return url_finale


def estrai_link_da_hub_avanzato(html_hub, nome_cartella, url_hub_corrente):
    """
    Estrae i link dall'Hub applicando il controllo rigido sul '2026'
    per le pagine HTML, escludendo automaticamente i duplicati senza anno e i vecchi archivi.
    """
    soup = BeautifulSoup(html_hub, 'html.parser')
    links_scoperti = set()

    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href'].strip()

        # 1. Filtro Lingua
        if "/en/" in href.lower() or "?language=en" in href.lower() or "lang=en" in href.lower():
            continue

        # 2. URL Assoluto
        if href.startswith('/'):
            href = "https://www.diem.unisa.it" + href

        # 3. Normalizzazione
        href_normalizzato = normalizza_url_unisa(href)

        # 4. Evitiamo loop sulla pagina Hub stessa
        if not href_normalizzato or href_normalizzato == url_hub_corrente or href_normalizzato == "https://www.diem.unisa.it":
            continue

        is_pdf = href_normalizzato.lower().endswith('.pdf')

        # 5. FILTRI SPECIFICI PER CARTELLA
        if nome_cartella == "bandi":
            if is_pdf:
                # I PDF nell'hub 2026 sono validi a prescindere dal nome del file
                links_scoperti.add(href_normalizzato)
            else:
                # 🎯 IL TUO CONTROLLO RIGIDO: Passa solo se il CMS vi ha propagato il "2026"
                if "2026" in href_normalizzato:
                    if "bando=" in href_normalizzato or "idConcorso=" in href_normalizzato:
                        links_scoperti.add(href_normalizzato)
        else:
            # Per news ed eventi accettiamo i dettagli basati su parametri o PDF
            if "/dettaglio/" in href_normalizzato.lower() or "?" in href_normalizzato or is_pdf:
                links_scoperti.add(href_normalizzato)

    return list(links_scoperti)

def estrai_allegati_pdf(html_content, url_pagina_sorgente):
    """
    Scansiona l'HTML di una pagina di dettaglio per trovare tutti i link ai file PDF.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    pdf_trovati = set()

    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href'].strip()

        if href.lower().split('?')[0].endswith('.pdf'):
            pdf_url_assoluto = urljoin(url_pagina_sorgente, href)
            pdf_url_pulito = normalizza_url_unisa(pdf_url_assoluto)

            if pdf_url_pulito:
                pdf_trovati.add(pdf_url_pulito)

    return list(pdf_trovati)

In [ ]:
# Ogni chiave è esattamente la cartella_origine presente nei metadati dei tuoi chunk
MANIFEST_CONFIG: Dict[str, str] = {
    "international": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_international_update.json",
    "docenti": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_docenti_update.json",
    "contatti": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_contatti_update.json",
    "Magistrale": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_offerta_formativa_update.json",
    "Triennale": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_offerta_formativa_update.json",
    "Dottorato": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_offerta_formativa_update.json",
    "bandi": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_bandi_update.json",
    "TerzaMissione": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_TerzaMissione_update.json",
    "news": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_news_update.json",
    "eventi": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_news_update.json",
    "ricerca": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_ricerca_update.json",
    "alternanza_scuola_lavoro": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_alternanza_scuola_lavoro_update.json",
    "focus_didattica": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_focus_didattica_update.json",
    "dipartimento": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_dipartimento_update.json",
    "consiglio_didattico": "/content/drive/Shareddrives/LLM/aggiornamenti/metadata_manifest/manifest_consiglio_didattico_update.json"
}

def load_all_manifests(config: Dict[str, str]) -> Dict[str, dict]:
    """
    Carica in memoria i file JSON associati a ciascuna cartella_origine.
    Restituisce un dizionario: cartella_origine -> contenuto_json_del_manifest
    """
    loaded_map: Dict[str, dict] = {}

    for cartella_nome, file_path in config.items():
        loaded_map[cartella_nome] = {} # Inizializzazione di fallback vuota

        if os.path.exists(file_path):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    loaded_manifest = json.load(f)
                    loaded_map[cartella_nome] = loaded_manifest
                print(f"✅ Manifest per cartella '{cartella_nome}' caricato correttamente.")
            except json.JSONDecodeError as e:
                print(f"❌ Errore di sintassi JSON nel file '{file_path}': {e}")
        else:
            print(f"⚠️ Avviso: Il file manifest non è stato trovato in: {file_path}")

    return loaded_map

# Caricamento della mappa globale all'avvio
manifests_map: Dict[str, dict] = load_all_manifests(MANIFEST_CONFIG)



# Mappa: cartella_origine -> Lista di chiavi di metadati rilevanti da preservare/ereditare
metadati_comuni = ["keywords", "questions"]

CARTELLA_METADATA_MAP = {
    "alternanza_scuola_lavoro": ["title", "year", "state"],
    "consiglio_didattico": ["commission", "membri"],
    "dipartimento": ["commission", "struttura", "membri"],
    "docenti": ["prof_name", "prof_surname", "courses"],
    "focus_didattica": ["livello", "corso", "curriculum"],
    "bandi": ["scadenza", "anno", "stato", "tipologia", "target", "dipartimento"],
    "contatti": ["nominativo", "ruolo", "ufficio"],
    "eventi": ["titolo","data","luogo"],
    "news": ["titolo","data","luogo"],
    "TerzaMissione": ["nome_progetto", "tipologia", "durata", "responsabile"],
    "ricerca": ["research_area", "responsible_person", "research_category", "project_period"],
    "international": ["programma", "destinatario", "paese", "universita_partner", "requisiti_linguistici", "data_scadenza", "data_stipula", "posti_disponibili", "coordinatore_diem"],
    "Magistrale": ["course_type", "course_name", "course_class", "academic_year", "category"],
    "Triennale": ["course_type", "course_name", "course_class", "academic_year", "category"],
    "Dottorato": ["course_type", "course_name", "course_class", "academic_year", "category"]
}


COLLECTIONS = [
    "docenti",
    "bandi",
    "international",
    "iniziative_ricerca_alternanza",
    "offerta_formativa",
    "dipartimento"
]

def get_unique_sources_and_metadata(client: QdrantClient, collection_name: str):
    """
    Estrae i source unici scorrendo i payload della collezione ed evitando
    di scaricare i vettori (molto leggero).
    """
    sources_map = {}
    next_page_offset = None

    # Continua a scorrere finché ci sono record nella collezione
    while True:
        points, next_page_offset = client.scroll(
            collection_name=collection_name,
            limit=1000,              # Recupera a blocchi di 1000 per efficienza
            with_payload=True,       # Ci serve il payload per i metadati
            with_vectors=False,      # CRITICO: False per non saturare la RAM
            offset=next_page_offset
        )

        for point in points:
            hit_payload = point.payload
            # Gestione fallback se LangChain incapsula in 'metadata'
            metadata = hit_payload.get("metadata", hit_payload)

            source = metadata.get("source")
            if not source:
                continue

            doc_type = metadata.get("doc_type")
            if doc_type == "pdf":
                continue

            # Se abbiamo già incontrato questo source, saltiamo (evita duplicati)
            if source in sources_map:
                continue

            sources_map[source] = {
                "cartella_origine": metadata.get("cartella_origine"),
                "hash": metadata.get("hash")
            }

        # Se non c'è una pagina successiva, interrompi il loop
        if next_page_offset is None:
            break

    return sources_map


BACKUP_BASE_DIR = (
    "/content/drive/Shareddrives/LLM/aggiornamenti"
)


def prepara_cartelle_backup(collection_name: str) -> tuple:
    """Crea la gerarchia di cartelle sul Drive se non esiste e restituisce i percorsi."""
    collezione_dir = os.path.join(BACKUP_BASE_DIR, collection_name)
    vecchio_dir = os.path.join(collezione_dir, "vecchio")
    nuovo_dir = os.path.join(collezione_dir, "nuovo")

    os.makedirs(vecchio_dir, exist_ok=True)
    os.makedirs(nuovo_dir, exist_ok=True)

    return vecchio_dir, nuovo_dir


def pulisci_nome_file(url: str) -> str:
    """Rende un URL un nome file valido sostituendo i caratteri speciali."""
    return re.sub(r"[^a-zA-Z0-9]", "_", url.replace("https://", "").replace("http://", ""))[:100]

def ottieni_vecchi_chunks_ordinati(client: QdrantClient, collection_name: str, source_url: str):
    """
    Recupera tutti i chunk esistenti per un dato source per poterne analizzare i metadati esistenti.
    """
    points, _ = client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[models.FieldCondition(key="metadata.source", match=models.MatchValue(value=source_url))]
        ),
        limit=500, # Regola in base al numero massimo atteso di chunk per singola pagina
        with_payload=True,
        with_vectors=False
    )

    # Ordiniamo la lista di punti in base al metadato chunk_index salvato nel payload
    # Usiamo un get con fallback a 0 o a x.id in caso di record vecchi senza indice
    points_ordinati = sorted(
        points,
        key=lambda x: x.payload.get("metadata", {}).get("chunk_index", 0)
    )

    return points_ordinati



def correggi_json_generale(modello, chunk_originale, json_rotto, errore, campi_richiesti_str, manifest):
    """
    Invia il chunk originale e il JSON malformato per una riparazione accurata,
    mantenendo lo schema dei campi dinamico in base alla cartella attuale.
    """
    ensure_ollama_running()
    recover_prompt = f"""
    Il JSON che hai generato è incompleto o sintatticamente errato.
    Errore rilevato dal parser: {errore}

    JSON DA RIPARARE:
    {json_rotto}

    TESTO ORIGINALE DI RIFERIMENTO (Usa questo per completare o correggere le informazioni):
    {chunk_originale}

    ATTIVITÀ:
    1. Sei un estrattore di dati puro. Non riassumere, non parafrasare e non commentare il testo.
       Il tuo unico output deve essere l'oggetto JSON richiesto descritto di seguito.
    2. Completa o correggi il JSON basandoti ESCLUSIVAMENTE sul testo originale di riferimento.
    3. Rispondi ESCLUSIVAMENTE con il JSON corretto, racchiuso tra parentesi graffe.
    4. Il tuo output deve contenere ESCLUSIVAMENTE le chiavi richieste elencate qui sotto.
    5. Se un'informazione non è presente nel testo originale, assegna il valore null alla chiave corrispondente. Non usare placeholder.

    CHIAVI OBBLIGATORIE DEL JSON DA RESTITUIRE:
    {campi_richiesti_str}
    "Le questions devono apparire come se fatte da uno studente e il contenuto del testo può fornire la risposta.\n"

    ESEMPIO DI COMPILAZIONE:
    {manifest}

    Assicurati che l'oggetto JSON creato sia sintatticamente perfetto e valido per json.loads().
    """
    print("   🔧 RECOVER: Invocazione del prompt di riparazione basata sul contesto...")
    return modello.invoke(recover_prompt).content.strip()


def estrai_metadati_con_llm(llm, testo_chunk, cartella, chiavi_specifiche, metadati_comuni, manifest):
    """
    Chiede al modello configurato in Ollama di compilare i metadati
    basandosi sul testo del chunk.
    Implementa una logica di retry con un prompt di correzione specifico
    e dinamico in base alla cartella d'origine.
    """
    # Verifichiamo preventivamente che il server locale Ollama risponda
    ensure_ollama_running()

    # Uniamo le chiavi specifiche della cartella a quelle semantiche comuni
    chiavi_da_estrarre = chiavi_specifiche + metadati_comuni

    # Prepariamo l'elenco dei campi formattato per i prompt
    campi_str = "\n".join([f"- {chiave}" for chiave in chiavi_da_estrarre])

    # Trasformiamo il dizionario in stringa JSON pulita ed esplicita per l'LLM
    manifest_str = json.dumps(manifest, indent=2, ensure_ascii=False) if manifest else "{}"

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "Sei un estrattore di dati puro. Non riassumere, non parafrasare e non commentare il testo.\n"
            "Analizza il testo del chunk fornito ed estrai esclusivamente le informazioni richieste.\n"
            "Rispondi SINTATTICAMENTE SOLO con un oggetto JSON valido.\n"
            "Non aggiungere introduzioni o spiegazioni. Non essere inutilmente verboso\n"
            "Se un'informazione non è presente nel testo, inserisci null. Non aggiungere placeholder.\n\n"
            "Campi da estrarre (CHIAVI DEL JSON):\n{campi}.\n"
            "Esempio di riempimento:\n{manifest}.\n"
            "Utilizza il campo descrizione solo per capire come riempire i campi, non produrre in uscita una descrizione.\n"
            "Il campo questions non è presente nel manifest ma devi crearle. Le questions devono apparire come se fatte da uno studente e il contenuto del testo può fornire la risposta.\n"
        )),
        ("human", "Testo del Chunk:\n---\n{testo}\n---")
    ])

    catena = prompt | llm

    max_retries = 1  # Tentativi di correzione consentiti
    tentativo = 0
    successo = False
    dati_estratti = None

    try:
        # Eseguiamo la prima chiamata standard
        print(f"   🧠 Invocazione iniziale LLM per estrazione metadati [{cartella}]...")
        response = catena.invoke({"campi": campi_str, "manifest": manifest_str, "testo": testo_chunk}).content.strip()
    except Exception as e:
        print(f"   ❌ Errore critico di comunicazione con Ollama: {e}")
        return {chiave: None for chiave in chiavi_da_estrarre}

    # Loop di validazione e retry in caso di errore di parsing sintattico
    while tentativo <= max_retries and not successo:
        try:
            # 1. Cerchiamo tutto quello che sta tra la prima { e l'ultima } per eliminare eventuale testo residuo
            match = re.search(r'(\{.*\})', response, re.DOTALL)

            if match:
                json_pulito = match.group(1)
                # Tentativo di decodifica del JSON
                dati_estratti = json.loads(json_pulito)
                # -------------------------------------------------------------
                # 🔧 FLATTENING FORZATO: Estraiamo solo ciò che ci serve ovunque si trovi
                # -------------------------------------------------------------
                flat_data = {}
                def estrai_chiavi_ricorsivamente(dizionario):
                    for k, v in dizionario.items():
                        if k in chiavi_da_estrarre:
                            flat_data[k] = v
                        elif isinstance(v, dict):
                            estrai_chiavi_ricorsivamente(v)

                if isinstance(dati_estratti, dict):
                    estrai_chiavi_ricorsivamente(dati_estratti)

                # Ricostruiamo il dizionario assicurandoci che ci siano tutte le chiavi (piatte)
                dati_estratti = {chiave: flat_data.get(chiave, None) for chiave in chiavi_da_estrarre}
                # -------------------------------------------------------------
                successo = True
            else:
                print(f"   ⚠️ Nessun blocco JSON rilevato nella risposta dell'LLM.")
                raise ValueError("Nessun blocco JSON trovato tra le parentesi graffe")

        except Exception as e:
            tentativo += 1
            if tentativo <= max_retries:
                print(f"   ⚠️ Errore JSON al tentativo {tentativo}: {e}. Avvio procedura di recupero...")
                # Chiamata alla funzione di correzione passando lo schema dinamico delle chiavi
                response = correggi_json_generale(llm, testo_chunk, response, str(e), campi_str, manifest_str)
            else:
                print(f"   ❌ Fallito recupero dell'output strutturato dopo {max_retries} tentativi per la cartella: {cartella}")

    # Se tutti i tentativi falliscono (es. timeout o allucinazioni ripetute della forma del JSON)
    if not successo or dati_estratti is None:
        print(f"   ⚠️ Fallback: Generazione di dizionario vuoto/nullo per preservare la transazione.")
        return {chiave: None for chiave in chiavi_da_estrarre}

    return dati_estratti


def scarica_contenuto_sicuro(url):
    """
    Scarica il contenuto di un URL camuffando la richiesta ed eseguendo
    dei tentativi automatici (retry) in caso di disconnessione dal server.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    for tentativo in range(3):
        try:
            # Richiesta con timeout per evitare blocchi infiniti
            response = requests.get(url, headers=headers, timeout=12)
            response.raise_for_status()

            # Un piccolo break di 0.5 secondi per non sovraccaricare il server di UNISA
            time.sleep(0.5)
            return response.text

        except Exception as e:
            print(f"  ⚠️ [Tentativo {tentativo + 1}/3] Errore di connessione per {url}: {e}")
            if tentativo < 2:
                time.sleep(3) # Aspetta 3 secondi prima di riprovare
            else:
                print(f"  ❌ Impossibile scaricare {url} dopo 3 tentativi. Salto la pagina.")
                return None

def update_vector_db_custom():
    for collection_name in COLLECTIONS:
        print(f"\n==================================================")
        print(f"--- 📂 Elaborazione collezione: {collection_name} ---")
        print(f"==================================================")

        # Inizializza il record manager specifico per la collezione
        record_manager = SQLRecordManager(
            namespace=f"qdrant/{collection_name}",
            db_url=RECORD_MANAGER_DB_URL
        )
        record_manager.create_schema()

        # Prepara e ottieni le cartelle di backup sul Drive per questa collezione
        vecchio_dir, nuovo_dir = prepara_cartelle_backup(collection_name)

        # =================================================================
        # FASE 1: FOTOGRAFIA DELLO STATO ATTUALE
        # =================================================================
        existing_sources = get_unique_sources_and_metadata(client, collection_name)

        # Istanzia il vector store di langchain associato alla collezione corrente
        vector_store = QdrantVectorStore(
            client=client,
            collection_name=collection_name,
            embedding=embeddings
        )

        # =================================================================
        # FASE 2: CONSOLIDAMENTO DELLA TO-DO LIST (DB Locale + Scraping)
        # =================================================================
        sources_to_process = {}

        # A. Inseriamo tutto ciò che è già nel DB
        for src_url, src_info in existing_sources.items():
            sources_to_process[src_url] = {
                "cartella_origine": src_info["cartella_origine"],
                "old_hash": src_info.get("hash")
            }

        # B. Rilevamento tramite Scraping per le cartelle abilitate
        cartelle_scraping = [c for c, h in HUBS_MAP.items() if h["collezione"] == collection_name]


        if cartelle_scraping:
            print(f"🌐 Rilevato Scraping attivo per le cartelle: {cartelle_scraping}")
            for cartella in cartelle_scraping:
                for hub_url in HUBS_MAP[cartella]["urls"]:
                    print(f"  🔍 Scaricamento Hub: {hub_url}...")
                    html_hub = scarica_contenuto_sicuro(hub_url)
                    if not html_hub:
                        continue

                    links_dettaglio = estrai_link_da_hub_avanzato(html_hub, cartella, hub_url)

                    for url_dettaglio in links_dettaglio:
                        if url_dettaglio not in sources_to_process:
                            sources_to_process[url_dettaglio] = {
                                "cartella_origine": cartella,
                                "old_hash": None # È un link nuovo
                            }
        else:
            print(f"📁 Modalità locale standard per questa collezione.")

        print(f"📋 Totale sorgenti iniziali da elaborare: {len(sources_to_process)}")

        # =================================================================
        # FASE 3: CICLO UNICO DI ESECUZIONE (con coda dinamica per i PDF)
        # =================================================================
        urls_queue = list(sources_to_process.keys())
        processed_urls = set()

        while urls_queue:
            source = urls_queue.pop(0)

            # Prevenzione loop su duplicati accodati
            if source in processed_urls:
                continue
            processed_urls.add(source)

            info = sources_to_process[source]
            cartella = info["cartella_origine"]
            old_hash = info["old_hash"]

            is_pdf = source.lower().split('?')[0].endswith('.pdf')

            # Download e conversione
            try:
                if is_pdf:
                    current_markdown = estrai_testo_PDF_STRUTTURATO(source)
                else:
                    html_content = scarica_contenuto_sicuro(source)
                    if not html_content:
                        continue
                    current_markdown = estrai_testo_STRUTTURATO(html_content)

                current_hash = calcola_hash(current_markdown)
            except Exception as e:
                print(f"❌ Errore nel processare {source}: {e}")
                continue

            # Controllo delle variazioni
            if current_hash == old_hash:
                print(f"✅ Sincronizzato [SKIP]: {source} non ha subito variazioni.")
                continue

            print(f"🔄 Rilevata variazione [UPDATE]: {source} in corso di aggiornamento...")

            # Discovery dinamica dei PDF (se la pagina HTML è variata, cerchiamo nuovi allegati)
            # Se sono qui è un nuovo link ottenuto con scraping o aggiornamento in place che potrebbe avere altri pdf allegati.
            if not is_pdf:
                pdf_links = estrai_allegati_pdf(html_content, source)
                for pdf_url in pdf_links:
                    if pdf_url not in sources_to_process:
                        print(f"  📎 Trovato nuovo allegato PDF: {pdf_url}")
                        sources_to_process[pdf_url] = {
                            "cartella_origine": cartella,
                            "old_hash": None
                        }
                        urls_queue.append(pdf_url)



            # Primo livello di splitting: divisione logica basata sui Markdown Header (#)
            markdown_documents = markdown_splitter.split_text(current_markdown)

            nuovi_sotto_chunks = []

            # Inizializziamo il contatore dei chunk a 0 per QUESTO specifico URL/Source
            nuovo_chunk_index = 0 # metadato chunk_index

            # Secondo livello di splitting: sotto-chunking bilanciato (Spacy / Recursive)
            for md_doc in markdown_documents:
                testo_sezione = md_doc.page_content

                # Filtro antirumore custom
                if not is_valid_chunk(testo_sezione):
                    continue

                # Esecuzione della tua funzione logica sul dimensionamento
                sotto_chunks = ottieni_sotto_chunks(testo_sezione)

                # Eredità e unione dei metadati associati all'intestazione Markdown
                for sub_chunk in sotto_chunks:
                    if not is_valid_chunk(sub_chunk):
                        continue
                    # Conserviamo provvisoriamente il testo e gli header rilevati dallo splitter markdown
                    nuovi_sotto_chunks.append({
                        "text": sub_chunk,
                        "md_metadata": md_doc.metadata,
                        "chunk_index": nuovo_chunk_index
                    })

                    # Incrementiamo per il prossimo sotto-chunk dello stesso URL
                    nuovo_chunk_index += 1


            # Valutazione quantitativa dei chunk (nuovi vs vecchi)
            vecchi_punti = ottieni_vecchi_chunks_ordinati(client, collection_name, source)
            num_vecchi = len(vecchi_punti)
            num_nuovi = len(nuovi_sotto_chunks)


            print(f"📊 Confronto quantitativo per {source}: Vecchi chunk = {num_vecchi} | Nuovi chunk = {num_nuovi}")

            # ─────────────── 💾 BACKUP STATO INIZIALE (VECCHIO) ───────────────
            timestamp = int(time.time())
            nome_base_file = f"{pulisci_nome_file(source)}_{timestamp}.json"

            vecchi_chunks_export = []
            for punto in vecchi_punti:
                vecchi_chunks_export.append(
                    {
                        "id": punto.id,
                        "payload": punto.payload,
                    }
                )

            # Salva il file JSON dei vecchi elementi nella cartella "vecchio"
            file_vecchio_path = os.path.join(vecchio_dir, nome_base_file)
            with open(file_vecchio_path, "w", encoding="utf-8") as f:
                json.dump(
                    vecchi_chunks_export, f, indent=2, ensure_ascii=False
                )
            print(
                f"💾 [BACKUP] {len(vecchi_chunks_export)} vecchi chunk salvati in: {file_vecchio_path}"
            )
            # ─────────────────────────────────────────────────────────────────

            final_documents_to_insert = []
            chiavi_specifiche = CARTELLA_METADATA_MAP.get(cartella, [])

            # --- SCENARIO A: Il numero di chunk è cresciuto significativamente -> Usiamo l'LLM ---
            if num_nuovi > num_vecchi:
                print(f"🧠 [STRATEGIA LLM] Rilevata espansione del testo. Avvio estrazione semantica con LLM")

                manifest = manifests_map.get(cartella, {})

                metadati_deterministici_fissi = {}
                cartelle_con_vincolo = ["Magistrale", "Triennale", "Dottorato"]
                campi_da_ereditare_sempre = ["course_type", "course_name"]

                if cartella in cartelle_con_vincolo and num_vecchi > 0:
                    # Estraiamo i valori dal primo chunk vecchio disponibile come riferimento stabile
                    primo_payload = vecchi_punti[0].payload.get("metadata", vecchi_punti[0].payload)
                    metadati_deterministici_fissi = {
                        k: primo_payload.get(k) for k in campi_da_ereditare_sempre if k in primo_payload
                    }

                chiavi_per_llm = [k for k in chiavi_specifiche if not (cartella in cartelle_con_vincolo and k in campi_da_ereditare_sempre)]

                for idx, nuovo_c in enumerate(nuovi_sotto_chunks):
                    print(f"  └─ Elaborazione chunk {idx+1}/{num_nuovi} via LLM...")

                    # Chiamata a Ollama per capire cosa c'è dentro questo specifico blocco di testo
                    metadati_llm = estrai_metadati_con_llm(llm, nuovo_c["text"], cartella, chiavi_per_llm, metadati_comuni, manifest)

                    # Costruzione del payload finale unendo le regole fisse e l'output dell'LLM
                    metadata_finale = {
                        "source": source,
                        "hash": current_hash,
                        "extracted_at": time.time(),
                        "doc_type": "pdf" if is_pdf else "html",
                        "cartella_origine": cartella,
                        "chunk_index": nuovo_c["chunk_index"],
                        "section": nuovo_c["md_metadata"].get("Header 1", ""),
                        "subsection": nuovo_c["md_metadata"].get("Header 2", ""),
                        "all_headers": list(nuovo_c["md_metadata"].values()) # Elenco ordinato degli header
                    }

                    # Integra i metadati estratti dall'LLM (sia comuni che specifici per cartella)
                    metadata_finale.update(metadati_llm)

                    if cartella in cartelle_con_vincolo:
                        metadata_finale.update(metadati_deterministici_fissi)


                    final_documents_to_insert.append(
                        Document(page_content=nuovo_c["text"], metadata=metadata_finale)
                    )

            # --- SCENARIO B: Il numero di chunk è inferiore o uguale -> Propaghiamo/Sostituiamo in modo lineare ---
            else:
                print(f"🔄 [STRATEGIA PROPAGAZIONE] Sostituzione lineare controllata (N. chunk stabili o ridotti).")

                # Estraiamo un dizionario di metadati "storici" dal primo chunk vecchio disponibile come fallback stabile
                fallback_metadata = {}
                if num_vecchi > 0:
                    primo_payload = vecchi_punti[0].payload.get("metadata", vecchi_punti[0].payload)
                    fallback_metadata = {k: primo_payload.get(k) for k in chiavi_specifiche + metadati_comuni}

                for idx, nuovo_c in enumerate(nuovi_sotto_chunks):
                    # Se esiste il chunk vecchio corrispondente allo stesso indice, prendiamo i suoi metadati precisi
                    if idx < num_vecchi:
                        payload_storico = vecchi_punti[idx].payload.get("metadata", vecchi_punti[idx].payload)
                        metadati_da_ereditare = {k: payload_storico.get(k) for k in chiavi_specifiche + metadati_comuni}
                    else:
                        # Caso limite se l'indice sfugge (usiamo il fallback del primo chunk)
                        metadati_da_ereditare = fallback_metadata

                    metadata_finale = {
                        "source": source,
                        "hash": current_hash,
                        "extracted_at": time.time(),
                        "doc_type": "pdf" if is_pdf else "html",
                        "cartella_origine": cartella,
                        "chunk_index": nuovo_c["chunk_index"],
                        "section": nuovo_c["md_metadata"].get("Header 1", ""),
                        "subsection": nuovo_c["md_metadata"].get("Header 2", ""),
                        "all_headers": list(nuovo_c["md_metadata"].values())
                    }

                    metadata_finale.update(metadati_da_ereditare)

                    final_documents_to_insert.append(
                        Document(page_content=nuovo_c["text"], metadata=metadata_finale)
                    )

            # ─────────────── 💾 SALVATAGGIO NUOVO STATO (NUOVO) ───────────────
            nuovi_chunks_export = []
            for doc in final_documents_to_insert:
                nuovi_chunks_export.append(
                    {"page_content": doc.page_content, "metadata": doc.metadata}
                )

            # Salva il file JSON dei nuovi elementi nella cartella "nuovo"
            file_nuovo_path = os.path.join(nuovo_dir, nome_base_file)
            with open(file_nuovo_path, "w", encoding="utf-8") as f:
                json.dump(nuovi_chunks_export, f, indent=2, ensure_ascii=False)
            print(
                f"💾 [BACKUP] {len(nuovi_chunks_export)} nuovi chunk pronti salvati in: {file_nuovo_path}"
            )
            # ─────────────────────────────────────────────────────────────────

            # Esecuzione della sostituzione sul database
            if final_documents_to_insert:
                # Esegui l'indicizzazione controllata da LangChain
                risultato_indicizzazione = index(
                    source_id_key="source",                       # Campo nei metadati che identifica la sorgente unica
                    cleanup="incremental",                               # "full" cancella tutto ciò che è legato al source e non è presente nella nuova lista
                    record_manager=record_manager,
                    vector_store=vector_store,
                    docs_source=final_documents_to_insert
                )
                print(f"🚀 Risultato SQL Record Manager: {risultato_indicizzazione}")

                # ─────────────── 🧠 AGGIORNAMENTO NAMED VECTORS (QUESTIONS) ───────────────
                print(f"🔮 Calcolo e inserimento Named Vectors per le domande di: {source}")

                # Recuperiamo i punti appena inseriti/aggiornati per questo specifico URL
                # per ottenere gli ID corretti generati o tracciati da LangChain
                punti_aggiornati, _ = client.scroll(
                    collection_name=collection_name,
                    scroll_filter=models.Filter(
                        must=[models.FieldCondition(key="metadata.source", match=models.MatchValue(value=source))]
                    ),
                    limit=len(final_documents_to_insert) + 10,
                    with_payload=True,
                    with_vectors=False
                )

                testi_da_embeddare = []
                mappa_vettori = []

                for punto in punti_aggiornati:
                    payload = punto.payload or {}
                    meta = payload.get("metadata", payload) # Gestione fallback struttura payload
                    questions = meta.get("questions", [])

                    if isinstance(questions, str):
                        questions = [questions]

                    if not isinstance(questions, list):
                        continue

                    # Pulizia stringhe vuote o non valide
                    questions = [
                        q.strip() for q in questions
                        if isinstance(q, str) and q.strip()
                    ]

                    if not questions:
                        continue

                    # Se c'è una sola domanda la duplichiamo, se sono di più prendiamo le prime 2
                    if len(questions) == 1:
                        questions_da_embeddare = [questions[0], questions[0]]
                    else:
                        questions_da_embeddare = questions[:2]

                    for idx, domanda in enumerate(questions_da_embeddare, start=1):
                        testi_da_embeddare.append(domanda)
                        # Tracciamo l'ID del punto e a quale named vector appartiene la domanda
                        mappa_vettori.append((punto.id, f"vector_question_{idx}"))

                # Se abbiamo domande da elaborare, generiamo gli embedding e aggiorniamo Qdrant
                if testi_da_embeddare:
                    print(f"    └─ Generazione di {len(testi_da_embeddare)} embedding per le domande...")
                    vettori_domande = embeddings.embed_documents(testi_da_embeddare)

                    vettori_per_punto = {}
                    for (punto_id, nome_vettore), vettore in zip(mappa_vettori, vettori_domande):
                        if punto_id not in vettori_per_punto:
                            vettori_per_punto[punto_id] = {}
                        vettori_per_punto[punto_id][nome_vettore] = vettore

                    points_to_update = [
                        models.PointVectors(
                            id=punto_id,
                            vector=vectors
                        )
                        for punto_id, vectors in vettori_per_punto.items()
                    ]

                    # Aggiornamento parziale dei vettori senza toccare il payload o il vettore principale ""
                    client.update_vectors(
                        collection_name=collection_name,
                        points=points_to_update
                    )
                    print(f"    ✅ Named Vectors aggiornati con successo per {len(points_to_update)} punti.")
                # ──────────────────────────────────────────────────────────────────────────
                print(f"🚀 Sincronizzazione completata con successo! {len(final_documents_to_insert)} chunk aggiornati su Qdrant.")


# Per eseguire la cella:
update_vector_db_custom()